# Give pUC19 a Golden Gate cloning site

Let's start with an annotated pUC19 plasmid from the bundled fixture and replace
its multiple cloning site (MCS) with a Golden Gate cloning site. You'll work on a
local branch and view the original and edited regions side by side in the
notebook.

Run the cells in order with Python 3.11 or newer and the Gen Python package
installed in your notebook kernel. The example creates a temporary file remote
on your computer, so it does not depend on a live server.


In [1]:
from pathlib import Path
import tempfile

import gen

work_dir = Path(tempfile.mkdtemp(prefix="gen-golden-gate-"))


## Get your own copy

Create a fresh local repository from the bundled annotated `puc19.gbk` fixture,
then clone it through a `file://` remote. This exercises the same clone workflow
without depending on a public repository's schema or availability. `gen.clone()`
returns an open `Repository` and accepts a `Path` directly.

To clone the published repository instead, use
`gen.clone("https://www.genhub.bio/api/repos/bvh/new-api-calls", path=work_dir / "clone")`.
For private repositories, set `GENHUB_API_KEY` in your environment or reuse
credentials saved by `gen remote login <remote-name>`.


In [2]:
remote_path = work_dir / "puc19-remote"
remote_path.mkdir()
remote_repo = gen.Repository(str(remote_path))
fixture_path = Path("puc19.gbk").resolve()
remote_repo.import_genbank(str(fixture_path), sample="pUC19")

repo_path = work_dir / "clone"
repo = gen.clone(remote_path.as_uri(), path=repo_path)
print("Your local copy:", repo_path)


Your local copy: /tmp/gen-notebook/clone


Preparing graph clone...
Opening the local graph remote...
Cloning graph history (attempt 1/1)...
Graph history cloned; preparing branch assets...
Checking asset 1/1: puc19.gbk
Checking asset 1/1: puc19.gbk
Materializing asset 1/1: puc19.gbk; 0 B written
Asset 1/1 ready; checksum verified: puc19.gbk (2.8 KiB staged, 2.8 KiB read this attempt)
Clone of branch 'main' is complete.


## Find the cloning site

Select the pUC19 sample and its sequence graph. Its annotations include an `MCS`
feature, so you can navigate to the cloning site by name without looking up
plasmid coordinates.


In [3]:
plasmid_sample = next(sample for sample in repo.samples if sample.name == "pUC19")
plasmid = plasmid_sample[0]
mcs = next(annotation for annotation in plasmid.annotations if annotation.name == "MCS")

original_view = plasmid.plot(rows=16, cols=120, detail="full", colors={"MCS": "#f9e2af"})
original_view.show(mcs, center=True)
original_view


                                                         puc19






cccagtcacgacgttgtaaaacgacggccagtGAATTCGAGCTCGGTACCCGGGGATCCTCTAGAGTCGACCTGCAGGCATGCAAGCTTggcgtaatcatggtcatagctgtttcctgtg
                                [══════════════════════════mcs══════════════════════════>
                                [══════════════════════════mcs══════════════════════════>




................|....................|...................|...................|...................|...................|..
               380                  400                 420                 440                 460                 480

## Try a Golden Gate design

Create and switch to a new branch in one call with `checkout(..., create=True)`.
Your `main` branch keeps the original plasmid while you try the new design.


In [4]:
design_branch = repo.checkout("golden-gate", create=True)


Checking out branch golden-gate


Replace the whole MCS with a short example cassette containing two BsaI
recognition sites (`GGTCTC` and its reverse complement, `GAGACC`), used in
[Golden Gate assembly](https://www.neb.com/en-us/products/e1601-nebridge-golden-gate-assembly-mix-basai-hf-v2).

`copy()` makes the `pUC19_golden_gate` sample, and passing the `mcs` annotation to
`replace()` swaps exactly the bases it covers, whatever its coordinates are. The edit is
recorded on your branch.


In [5]:
golden_gate_mcs = "GAATTCGGTCTCAAATGCATCATCATCATGCATTGAGACCAAGCTT"
edited_sample = plasmid_sample.copy("pUC19_golden_gate")
edited_sample[0].replace(mcs, golden_gate_mcs)


Locus([2d61f0fb[0:46][:]], strand='+')

## See what changed

Open the edited graph and highlight the replacement sequence. Compare it with
the original MCS view above: the new sample has an alternative route through
the Golden Gate cassette.


In [6]:
edited_plasmid = edited_sample[0]
replacement = edited_plasmid.search(golden_gate_mcs)[0]
edited_view = edited_plasmid.plot(detail="full", cols=80, colors={"MCS": "#f9e2af"})
edited_view.show(replacement)
edited_view


                                     puc19




─GAATTCGGTCTCAAATGCATCATCATCATGCATTGAGACCAAGCTT──ggcgtaatcatggtcatagctgtttcctgtg




 .....|...................................................|.....................
     400                                                 450

Switch back to `main` to see the original samples, then return to your design.
`checkout()` accepts either a branch name or the `Branch` object returned when
you created it. Without `create=True`, it switches to an existing branch.


In [7]:
repo.checkout("main")
print("On main:", [sample.name for sample in repo.samples])

repo.checkout(design_branch)
print("On golden-gate:", [sample.name for sample in repo.samples])


On main: ['pUC19']
On golden-gate: ['pUC19_golden_gate', 'pUC19']
Checking out branch main
Checking out branch golden-gate


Your Golden Gate design is saved on the local `golden-gate` branch. You can
continue editing it or return to `main` whenever you like. Nothing has been
pushed to GenHub.
